# **Proyek ETL Pipeline & Analisis Pemasaran Perbankan (Bank Marketing)**
## **Layer 3: Gold Layer (Business Aggregations, Data Mart & Pipeline Operationalization)**

Notebook ini merupakan tahap akhir dari arsitektur Medallion. Pada tahap ini dilakukan:
1. **Pemuatan Data dari Silver:** Mengambil data bersih dari `lakehouse/silver/bank_marketing`.
2. **Business Aggregations (Data Mart):** Membangun tabel agregasi analitik performa kampanye berdasarkan kelompok usia nasabah (`AgeGroup`) dan status konversi deposito (`y`).
3. **Penyimpanan ke Gold Layer & Ekspor BI:** Menyimpan tabel data mart ke Parquet (`lakehouse/gold/bank_marketing_summary`) dan file CSV tunggal.
4. **Panduan Operasional & Otomatisasi Pipeline:** Dokumentasi implementasi jadwal batch (3 menit), aturan dependensi DAG, penanganan kegagalan, dan strategi penanganan batch baru.

---
### **1. Inisialisasi Sesi Spark**

In [ ]:
# Instal pyspark jika dijalankan di Google Colab terpisah
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, round, col

# Inisialisasi Spark Session
spark = SparkSession.builder \
    .appName("BankMarketing_03_Gold") \
    .getOrCreate()

print("Spark Session untuk Layer Gold berhasil diinisialisasi.")

---
### **2. Membaca Data dari Layer Silver**

In [ ]:
silver_path = "lakehouse/silver/bank_marketing"

# Ambil data dari Silver
silver_data = spark.read.parquet(silver_path)

print(f"Data Silver dimuat dari: {silver_path}")
print(f"Total baris data Silver: {silver_data.count()}")
silver_data.printSchema()
silver_data.show(5)

---
### **3. Agregasi Bisnis (Business Data Mart: AgeGroup & Target Deposito)**

Membangun ringkasan analitik bisnis untuk mengukur efektivitas kampanye:
* `total_klien`: Total jumlah nasabah pada masing-masing segmen.
* `avg_saldo`: Rata-rata saldo tahunan nasabah.
* `avg_durasi_telepon`: Rata-rata durasi percakapan telepon kampanye (dalam detik).

In [ ]:
# Buat agregasi ringkas sebagai data mart Gold
gold_agg = silver_data.groupBy("AgeGroup", "y") \
    .agg(
        count("*").alias("total_klien"),
        round(avg("balance"), 2).alias("avg_saldo"),
        round(avg("duration"), 2).alias("avg_durasi_telepon")
    ) \
    .orderBy("AgeGroup", "y")

print("=== Tabel Data Mart Gold (Summary Kampanye Bank Marketing) ===")
gold_agg.show()

---
### **4. Penyimpanan ke Gold Layer & Ekspor CSV Tunggal**

In [ ]:
# Simpan ke Gold Layer (format Parquet)
gold_path = "lakehouse/gold/bank_marketing_summary"
gold_agg.write.mode("overwrite").parquet(gold_path)

print(f"Layer Gold Berhasil Terbentuk di: {gold_path}")

# Verifikasi membaca kembali data Gold
print("\nVerifikasi Data Gold dari Parquet:")
spark.read.parquet(gold_path).show()

# Ekspor ke CSV Tunggal untuk BI Dashboard (Tableau, PowerBI, Google Looker Studio)
gold_export_path = "lakehouse/exports/gold_export"
gold_agg.coalesce(1).write.mode("overwrite").option("header", "true").csv(gold_export_path)
print(f"Ekspor Gold ke CSV tunggal berhasil di: {gold_export_path}")

---
### **5. Panduan Operasional & Otomatisasi Pipeline Medallion (Jobs & Workflow)**

#### **1. Pembuatan Job & Aturan Dependensi (DAG)**
* **Task 1: Ingest_Bronze**
  * Menjalankan notebook `01_EDA_dan_Bronze_Layer.ipynb`.
  * Mengambil micro-batch 150 baris secara incremental dengan pencatatan checkpoint.
  * Tanpa dependensi upstream (root task).
* **Task 2: Clean_Silver**
  * Menjalankan notebook `02_Silver_Layer.ipynb`.
  * Dependensi: Berhasil menyelesaikan **Task 1: Ingest_Bronze**.
  * Membersihkan duplikasi hasil append Bronze dan menambahkan `AgeGroup`.
* **Task 3: Aggregate_Gold**
  * Menjalankan notebook `03_Gold_Layer.ipynb`.
  * Dependensi: Berhasil menyelesaikan **Task 2: Clean_Silver**.
  * Memperbarui data mart analitik bisnis.
* **Aturan Kegagalan:** Jika salah satu tahap gagal (misal kegagalan koneksi di Bronze), tahap hilir (Silver dan Gold) otomatis dibatalkan (*skipped*).

#### **2. Parameterisasi & Kebijakan Retry**
* **Parameters:**
  * `batch_size`: `150` (ukuran baris per batch).
  * `source_path`: `bank_marketing.csv`.
  * `checkpoint_file`: `lakehouse/bronze/_checkpoint.json`.
* **Compute:** Single-node cluster dengan runtime Spark (Databricks Runtime LTS / Apache Spark 3.x).
* **Retry & Timeout:**
  * **Max Retries:** `2` kali percobaan ulang jika terjadi kendala jaringan/I/O.
  * **Retry Interval:** `5` menit jeda antar percobaan.
  * **Timeout:** Maksimum `30` menit per task.

#### **3. Penjadwalan (Scheduling) & Mekanisme 3 Menit**
* **Metode:** Scheduled Trigger (Cron).
* **Cron Expression:** `*/3 * * * *` (berjalan secara otomatis setiap 3 menit).
* **Mekanisme Reset:** Menggunakan state offset pada file checkpoint. Begitu seluruh data terambil, pipeline otomatis me-reset offset ke 0 untuk siklus berikutnya tanpa intervensi manual.

#### **4. Monitoring & Analisis Kegagalan**
* **Verifikasi Run:** Memeriksa grafik DAG pada orchestration platform (Databricks Workflows / Airflow) untuk memastikan status *SUCCESS* (hijau).
* **Analisis Log & Alerting:** Jika task gagal, notifikasi otomatis dikirim ke email tim data engineering dan log dapat dianalisis di tab **Standard Error / Run Logs**.

#### **5. Sifat Incremental & Siklus Medallion**
* **Bronze:** Menggunakan mode `.mode("append")` untuk mempertahankan histori utuh setiap batch beserta timestamp ingest.
* **Silver:** Menggunakan `.dropDuplicates()` untuk menyaring duplikasi antar-batch atau antar-siklus.
* **Gold:** Menggunakan `.mode("overwrite")` untuk menyajikan ringkasan analitik terkini bagi para pengambil keputusan.

In [ ]:
print("Pipeline Medallion Bank Marketing (Bronze Incremental -> Silver Clean -> Gold Mart) berhasil dioperasionalkan!")